# B2-022-probabilistic-latent-models — Practice p05 — Solution

**Type:** mc · **Difficulty:** core · **Concepts:** gaussian-reparameterization

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Solution

**Answer: C.**
Sampling $z\sim N(\mu,\operatorname{diag}e^{\ell})$ with a sampler is not a differentiable function of $(\mu,\ell)$.
Writing $z=\mu+\exp(\tfrac12\ell)\odot\varepsilon$ with parameter-free $\varepsilon\sim N(0,I)$ keeps the distribution of $z$ the same while making $z$ (and hence the Monte Carlo reconstruction term) a differentiable function of $\mu$, $\ell$, and the encoder weights: $\partial z/\partial\mu=1$ and $\partial z/\partial\ell=\tfrac12e^{\ell/2}\varepsilon$.

Rejected options:

- **A.** $\varepsilon$ is fixed noise drawn independently of all parameters; it is never learned or updated.
- **B.** The closed-form KL is already an explicit smooth function of $\mu$ and $\ell$, so it needs no trick to be differentiable.
- **D.** $\log p(x)$ remains intractable; the trick only gives low-variance gradients of the ELBO, which is still the objective.
- **E.** The decoder is a neural network and is differentiable in its own weights regardless; the trick concerns gradients reaching the *encoder* through the sample.

In [ ]:
import torch

mu = torch.tensor([[0.3, -0.2]], requires_grad=True)
logvar = torch.tensor([[0.0, -1.0]], requires_grad=True)
eps = torch.randn(1, 2, generator=torch.Generator().manual_seed(20260927))
z = mu + torch.exp(0.5 * logvar) * eps
(z ** 2).sum().backward()
with torch.no_grad():
    blocked = torch.normal(mu, torch.exp(0.5 * logvar))   # direct sampling: no graph to mu/logvar
ANSWER = "C"
print(mu.grad, logvar.grad, blocked.requires_grad)

### Answer check

In [ ]:
assert torch.allclose(mu.grad, 2 * z.detach(), atol=1e-6, rtol=1e-6)
assert torch.allclose(logvar.grad, 2 * z.detach() * 0.5 * torch.exp(0.5 * logvar.detach()) * eps, atol=1e-6, rtol=1e-6)
assert eps.grad is None and not eps.requires_grad
assert blocked.requires_grad is False
assert ANSWER == "C"